# Face Anti-Spoofing — SiW (Spoof in the Wild) Dataset

This notebook trains and tests a **live vs. spoof** detector on the SiW
dataset (165 subjects, live videos + print/replay spoof attacks).

It reuses the same hand-crafted feature pipeline (frequency / boundary /
temporal cues) and the same classical + deep-learning models as your
FaceForensics++ deepfake notebook, just retargeted at a different
question: **is this a live camera feed, or a presentation attack
(printed photo / replayed video)?**

Run cells top to bottom. GPU runtime required (Runtime → Change runtime
type → T4 GPU).

## Stage 0 — Setup

In [ ]:
import os, sys
import torch
if not torch.cuda.is_available():
    raise SystemExit("❌ No GPU. Menu: Runtime → Change runtime type → T4 GPU → Save. Then re-run from the top.")
print("GPU:", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount("/content/drive")
!pip -q install kaggle timm
!pip -q install facenet-pytorch --no-deps
print("✅ setup done")

### Config (don't change unless something runs out of time/disk)

In [ ]:
import os, sys, json, time, random, zipfile, shutil, glob, math, zlib, warnings, urllib.request
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
cv2.setNumThreads(2)

CFG = dict(
    KAGGLE_SLUG   = "kokomi3012/siw-dataset",
    WORK          = "/content/work",                         # local scratch (lost on disconnect)
    DRIVE         = "/content/drive/MyDrive/siw_project",     # saved results (kept)
    MAX_SUBJECTS  = None,     # set e.g. 40 to subsample subjects for a faster first run, or None for all
    T             = 8,        # frames per clip (consecutive)
    SIZE          = 224,      # face-crop size
    CLIPS_PER_VIDEO = 2,
    SEED          = 42,
    NJOBS         = max(1, os.cpu_count() or 1),
    # Official-ish SiW protocol: first ~90 subjects train, rest test (we also carve a val split
    # out of train). SiW's own Protocol 1 splits by subject ID; we follow that spirit.
    TRAIN_FRAC    = 0.6,
    VAL_FRAC      = 0.15,     # taken out of the train pool
)
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"])

def P(*a):
    """path under WORK"""
    return os.path.join(CFG["WORK"], *a)
def D(*a):
    """path under Drive"""
    return os.path.join(CFG["DRIVE"], *a)

for d in [P("dl"), P("videos"), P("clips"), P("models"), D("results"), D("checkpoints")]:
    os.makedirs(d, exist_ok=True)
HAVE_CLIPS = os.path.exists(D("checkpoints", "clips_clean.tar"))   # True on a resumed run -> no re-download needed
print("work:", CFG["WORK"], "| drive:", CFG["DRIVE"], "| resuming from Drive checkpoint:", HAVE_CLIPS)

## Stage 1 — Get the dataset from Kaggle

In [ ]:
import getpass, pathlib
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
tok = "" if HAVE_CLIPS else getpass.getpass("Paste Kaggle API token (or just press Enter to upload kaggle.json instead): ").strip()
if HAVE_CLIPS:
    print("Drive checkpoint found -> Kaggle not needed, skipping.")
elif tok:
    os.environ["KAGGLE_API_TOKEN"] = tok
    p = pathlib.Path("~/.kaggle/access_token").expanduser(); p.write_text(tok); p.chmod(0o600)
else:
    from google.colab import files
    up = files.upload()
    src = list(up)[0]
    dst = os.path.expanduser("~/.kaggle/kaggle.json")
    shutil.move(src, dst); os.chmod(dst, 0o600)
KS = CFG["KAGGLE_SLUG"]
if not HAVE_CLIPS:
    !kaggle datasets files {KS} | head -8
    print("↑ if you see a file list, Kaggle login works. If you see 401/403, redo the token step.")

In [ ]:
WORKDL = P("dl")
if HAVE_CLIPS:
    print("Drive checkpoint found -> skipping big download.")
else:
    if not glob.glob(WORKDL + "/*.zip"):
        !kaggle datasets download -d {KS} -p {WORKDL}
    if not glob.glob(WORKDL + "/*.zip"):
        raise SystemExit(f"❌ Download failed. Open https://www.kaggle.com/datasets/{KS} in your browser while logged in, "
                         "check it exists, redo the token step, and run this cell again. Paste the error text to Claude if it persists.")
    ZIP = glob.glob(WORKDL + "/*.zip")[0]
    print("zip:", ZIP, round(os.path.getsize(ZIP) / 1e9, 1), "GB")
    !df -h /content | tail -1

In [ ]:
# Unzip. SiW mirrors on Kaggle typically preserve the original
# Train/Test or Live/Spoof folder layout with .mov/.mp4 files named
# like SubjectID_SensorID_TypeID_MediumID_SessionID.{mov,mp4}
if not HAVE_CLIPS:
    EXDIR = P("videos_raw")
    os.makedirs(EXDIR, exist_ok=True)
    if not glob.glob(EXDIR + "/**/*", recursive=True):
        with zipfile.ZipFile(ZIP) as z:
            z.extractall(EXDIR)
    print("extracted to:", EXDIR)
    !find {EXDIR} -maxdepth 3 -type d | head -30
    print("---- sample video files ----")
    !find {EXDIR} -iname "*.mov" -o -iname "*.mp4" | head -10

## Stage 2 — Index videos and assign live/spoof labels + subject-based splits

SiW filenames encode `SubjectID_SensorID_TypeID_MediumID_SessionID`.
**TypeID 1 = live**, TypeID 2/3 = spoof (print / replay attacks).
If your copy of the dataset instead uses `Live/` and `Spoof/` folders,
the indexer below falls back to folder-name detection automatically —
check the printed label counts after running this cell to make sure
the labels came out sane (roughly balanced, not all-one-class).

In [ ]:
def find_all_videos(root):
    exts = (".mov", ".mp4", ".avi", ".MOV", ".MP4")
    return [os.path.join(dp, f) for dp, _, fs in os.walk(root) for f in fs if f.endswith(exts)]

def label_from_name(path):
    """Return (subject_id, label) where label=0 (live) or 1 (spoof)."""
    name = os.path.basename(path)
    stem = os.path.splitext(name)[0]
    parts = stem.split("_")
    # Pattern A: SubjectID_SensorID_TypeID_MediumID_SessionID (official SiW naming)
    if len(parts) >= 3 and parts[0].isdigit():
        subj = parts[0]
        type_id = parts[2] if len(parts) > 2 and parts[2].isdigit() else None
        if type_id is not None:
            return subj, (0 if type_id == "1" else 1)
    # Pattern B: fall back to folder name containing "live"/"spoof"/"real"/"fake"/"attack"
    lower = path.lower()
    subj = parts[0] if parts and parts[0].isdigit() else "unk"
    if any(k in lower for k in ["live", "real"]):
        return subj, 0
    if any(k in lower for k in ["spoof", "fake", "attack", "print", "replay", "mask"]):
        return subj, 1
    return subj, None   # unknown -> caller should drop these

VIDEOS = find_all_videos(P("videos_raw")) if not HAVE_CLIPS else []
if not HAVE_CLIPS:
    rows = []
    for v in VIDEOS:
        subj, lab = label_from_name(v)
        if lab is None:
            continue
        rows.append(dict(path=v, subject=subj, label=lab))
    VIDX = pd.DataFrame(rows)
    print("videos indexed:", len(VIDX), "| unlabeled/dropped:", len(VIDEOS) - len(VIDX))
    print(VIDX.label.value_counts().rename({0: "live", 1: "spoof"}))

    if CFG["MAX_SUBJECTS"]:
        keep_subj = sorted(VIDX.subject.unique())[: CFG["MAX_SUBJECTS"]]
        VIDX = VIDX[VIDX.subject.isin(keep_subj)].reset_index(drop=True)
        print("subsampled to", len(keep_subj), "subjects ->", len(VIDX), "videos")

    # Subject-disjoint split: no subject appears in more than one split
    # (this is what makes the test score meaningful -- otherwise the model
    # could just memorize faces instead of learning spoof cues)
    subjects = sorted(VIDX.subject.unique())
    rng = random.Random(CFG["SEED"]); rng.shuffle(subjects)
    n = len(subjects)
    n_train = int(n * CFG["TRAIN_FRAC"]); n_val = int(n * CFG["VAL_FRAC"])
    train_subj = set(subjects[:n_train])
    val_subj   = set(subjects[n_train:n_train + n_val])
    test_subj  = set(subjects[n_train + n_val:])
    def split_of(s):
        return "train" if s in train_subj else ("val" if s in val_subj else "test")
    VIDX["split"] = VIDX.subject.apply(split_of)
    VIDX["stem"] = VIDX.path.apply(lambda p: os.path.splitext(os.path.basename(p))[0])
    VIDX["cat"] = VIDX.label.map({0: "Live", 1: "Spoof"})
    VIDX.to_csv(P("videos_index.csv"), index=False)
    print(VIDX.groupby(["split", "cat"]).size().unstack(0).fillna(0).astype(int))

## Stage 3 — Face-crop clips (8 consecutive frames x 2 clips per video)

In [ ]:
class FaceDetector:
    """MTCNN if it installs cleanly, otherwise OpenCV Haar. Both return (cx, cy, side) or None."""
    def __init__(self):
        self.mtcnn, self.last_src = None, None
        try:
            import torch
            from facenet_pytorch import MTCNN
            self.mtcnn = MTCNN(keep_all=True, device="cuda" if torch.cuda.is_available() else "cpu", post_process=False)
            print("face detector: MTCNN")
        except Exception as e:
            print("MTCNN unavailable -> Haar fallback (", type(e).__name__, ")")
        self.haar = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
        self.stats = {"mtcnn": 0, "haar": 0, "none": 0}

    def _box(self, x1, y1, x2, y2, s):
        w, h = (x2 - x1) / s, (y2 - y1) / s
        return ((x1 + x2) / 2 / s, (y1 + y2) / 2 / s, max(w, h) * 1.3)

    def detect(self, bgr):
        H, W = bgr.shape[:2]
        s = min(1.0, 640.0 / max(H, W))
        small = cv2.resize(bgr, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else bgr
        if self.mtcnn is not None:
            try:
                boxes, _ = self.mtcnn.detect(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
                if boxes is not None and len(boxes):
                    b = max(boxes, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))
                    cx, cy, side = self._box(*b, 1.0)
                    self.last_src = "mtcnn"; self.stats["mtcnn"] += 1
                    return (cx / s, cy / s, side / s)
            except Exception:
                pass
        g = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
        fs = self.haar.detectMultiScale(g, 1.1, 5, minSize=(48, 48))
        if len(fs):
            x, y, w, h = max(fs, key=lambda f: f[2] * f[3])
            cx, cy, side = self._box(x, y, x + w, y + h, 1.0)
            self.last_src = "haar"; self.stats["haar"] += 1
            return (cx / s, cy / s, side / s)
        self.stats["none"] += 1
        return None

def read_clip(path, start, T):
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames = []
    for _ in range(T):
        ok, f = cap.read()
        if not ok: break
        frames.append(f)
    cap.release()
    return frames

def crop_resize(bgr, box, size):
    cx, cy, side = box
    s = int(round(side)); x1 = int(round(cx - side / 2)); y1 = int(round(cy - side / 2))
    x2, y2 = x1 + s, y1 + s
    H, W = bgr.shape[:2]
    pl, pt, pr, pb = max(0, -x1), max(0, -y1), max(0, x2 - W), max(0, y2 - H)
    if pl or pt or pr or pb:
        bgr = cv2.copyMakeBorder(bgr, pt, pb, pl, pr, cv2.BORDER_REPLICATE)
        x1 += pl; x2 += pl; y1 += pt; y2 += pt
    crop = bgr[y1:y2, x1:x2]
    interp = cv2.INTER_AREA if s > size else cv2.INTER_CUBIC
    return cv2.resize(crop, (size, size), interpolation=interp)

def process_video(path, cat, stem, split, det, label):
    T, S = CFG["T"], CFG["SIZE"]
    cap = cv2.VideoCapture(path); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    if n < T + 2: return []
    outdir = P("clips", "clean"); os.makedirs(outdir, exist_ok=True)
    rows = []
    for k, frac in enumerate((0.2, 0.6)[: CFG["CLIPS_PER_VIDEO"]]):
        frames = read_clip(path, int(frac * (n - T)), T)
        if len(frames) < T: continue
        box = det.detect(frames[T // 2])
        if box is None: continue
        crops = np.stack([crop_resize(f, box, S) for f in frames])[..., ::-1]   # BGR -> RGB
        cid = f"{cat}__{stem}__c{k}"
        np.save(f"{outdir}/{cid}.npy", np.ascontiguousarray(crops))
        rows.append(dict(clip_id=cid, video_id=f"{cat}__{stem}", method=cat,
                         label=int(label), split=split, variant="clean"))
    return rows

In [ ]:
META_CSV = P("meta_clean.csv")
if os.path.exists(D("checkpoints", "clips_clean.tar")) and not os.path.exists(META_CSV):
    print("restoring clips from Drive checkpoint ...")
    os.makedirs(P("clips"), exist_ok=True)
    !tar -xf {D("checkpoints", "clips_clean.tar")} -C {P("clips")}
    shutil.copy(D("checkpoints", "meta_clean.csv"), META_CSV)

if not os.path.exists(META_CSV):
    det = FaceDetector()
    vids = pd.read_csv(P("videos_index.csv"))
    rows = []
    for r in tqdm(vids.itertuples(), total=len(vids), desc="clips"):
        rows += process_video(r.path, r.cat, r.stem, r.split, det, r.label)
    print("detector usage:", det.stats)
    pd.DataFrame(rows).to_csv(META_CSV, index=False)
    # checkpoint to Drive (single big file = fast)
    !tar -cf {P("clips_clean.tar")} -C {P("clips")} clean
    shutil.copy(P("clips_clean.tar"), D("checkpoints", "clips_clean.tar"))
    shutil.copy(META_CSV, D("checkpoints", "meta_clean.csv"))
    os.remove(P("clips_clean.tar"))

META = pd.read_csv(META_CSV)
print(META.groupby(["split", "method"]).size().unstack(0).fillna(0).astype(int))
META_ALL = META.copy()   # kept for naming-compatibility with the feature/model code below

## Stage 4 — Hand-crafted image-processing features

Same three feature families as the deepfake notebook — they're generic
pixel-statistics cues, not deepfake-specific, so they transfer directly
to spotting print/replay spoofing:
- **F (frequency):** FFT/DCT radial power spectrum, spectral slope, JPEG
  blockiness, chroma spectra — catches the sub-pixel/moiré grid and
  re-compression artifacts of a printed photo or a screen replay.
- **B (boundary/blending):** gradient/noise/Laplacian energy, center-vs-
  outer ratios — catches flat/blurred edges typical of a flat print or a
  second-generation recording.
- **T (temporal):** optical flow magnitude/divergence/acceleration,
  flicker — catches the lack of natural micro-motion in a static photo,
  or display refresh flicker in a screen replay.

In [ ]:
import scipy.fft
S_ = CFG["SIZE"]; NB = 32; NBR = 12
_yy, _xx = np.indices((S_, S_)); _R = np.hypot(_yy - (S_ - 1) / 2, _xx - (S_ - 1) / 2)
_RB = np.minimum((_R / (S_ / 2) * NB).astype(int), NB)                 # radial bin (NB == outside)
_RHO = _R / (S_ / 2)
_RBB = np.minimum((_RHO * NBR).astype(int), NBR)                       # boundary-ring bins
_HAN = np.outer(np.hanning(S_), np.hanning(S_)).astype(np.float32)
_y1, _x1 = np.indices((112, 112)); _RHO112 = np.hypot(_y1 - 55.5, _x1 - 55.5) / 56.0
_CEN, _RING = _RHO112 < 0.5, (_RHO112 >= 0.5) & (_RHO112 < 0.95)

def _radial(P2, nb, rb):
    m = rb < nb
    return np.bincount(rb[m], P2[m], minlength=nb) / np.maximum(np.bincount(rb[m], minlength=nb), 1)

def _logspec(ch):
    F = np.fft.fftshift(np.fft.fft2((ch - ch.mean()) * _HAN))
    Pw = np.abs(F) ** 2
    return np.log10(Pw + 1e-8), Pw

def _band_peak(v):
    v = v - np.polyval(np.polyfit(np.arange(len(v)), v, 2), np.arange(len(v)))
    s = np.abs(np.fft.rfft(v * np.hanning(len(v))))[2:60]
    return float(s.max() / (s.mean() + 1e-6))

def _dct_feats(ch):
    b = ch.reshape(28, 8, 28, 8).transpose(0, 2, 1, 3).reshape(-1, 8, 8)
    d = scipy.fft.dctn(b - 128.0, axes=(1, 2), norm="ortho")
    L = np.log1p(np.abs(d)).mean(0); s = np.indices((8, 8)).sum(0)
    return [L[s == k].mean() for k in range(1, 15)], float((np.abs(d[:, 4:, 4:]) < 0.5).mean())

def frame_feats(rgb):
    f = {}
    ycc = cv2.cvtColor(rgb, cv2.COLOR_RGB2YCrCb).astype(np.float32)
    Y, Cr, Cb = ycc[..., 0], ycc[..., 1], ycc[..., 2]
    # ---- F: frequency
    P2, Pw = _logspec(Y)
    prof = _radial(P2, NB, _RB)
    for i, v in enumerate(prof): f[f"f_rad{i:02d}"] = v
    f["f_slope"] = np.polyfit(np.log(np.arange(4, NB) + 0.5), prof[4:], 1)[0]
    inside = _R < S_ / 2
    f["f_hf"] = np.log10(Pw[inside & (_R > 0.5 * S_ / 2)].sum() / Pw[inside].sum() + 1e-12)
    exp2d = prof[np.minimum(_RB, NB - 1)]
    msk = (_R >= 0.12 * S_ / 2) & inside
    r = (P2 - exp2d)[msk]; rr = _R[msk]
    f["f_pk_max"] = r.max(); f["f_pk_p999"] = np.percentile(r, 99.9)
    f["f_pk_frac"] = float((r > 3 * r.std()).mean()); f["f_pk_rad"] = rr[r.argmax()] / (S_ / 2)
    dY, zr = _dct_feats(Y)
    for i, v in enumerate(dY): f[f"f_dct{i + 1:02d}"] = v
    f["f_dct_zero"] = zr
    dxx = np.abs(np.diff(Y, axis=1)); dyy = np.abs(np.diff(Y, axis=0))
    f["f_blk_h"] = dxx[:, 7::8].mean() / (dxx.mean() + 1e-6); f["f_blk_v"] = dyy[7::8, :].mean() / (dyy.mean() + 1e-6)
    for nm, ch in (("cr", Cr), ("cb", Cb)):
        pc = _radial(_logspec(ch)[0], 8, np.minimum((_R / (S_ / 2) * 8).astype(int), 8))
        for i, v in enumerate(pc): f[f"f_{nm}_rad{i}"] = v
    f["f_band_row"] = _band_peak(Y.mean(1)); f["f_band_col"] = _band_peak(Y.mean(0))
    # ---- B: boundary / blending
    gx = cv2.Sobel(Y, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(Y, cv2.CV_32F, 0, 1, ksize=3)
    gm = np.hypot(gx, gy)
    nr = np.abs(Y - cv2.GaussianBlur(Y, (0, 0), 1.5)); lp = np.abs(cv2.Laplacian(Y, cv2.CV_32F))
    for nm, mp in (("gm", gm), ("nr", nr), ("lp", lp)):
        pr = _radial(mp, NBR, _RBB)
        for i, v in enumerate(pr): f[f"b_{nm}{i:02d}"] = v
        if nm == "gm": f["b_gm_step"] = float(np.abs(np.diff(pr)).max())
    cen, out = _RHO < 0.35, (_RHO > 0.75) & (_RHO < 1.0)
    f["b_dY"] = Y[cen].mean() - Y[out].mean(); f["b_dCr"] = Cr[cen].mean() - Cr[out].mean(); f["b_dCb"] = Cb[cen].mean() - Cb[out].mean()
    for nm, mp in (("gm", gm), ("nr", nr), ("lp", lp)):
        f[f"b_{nm}_ratio"] = mp[cen].mean() / (mp[out].mean() + 1e-6)
    return f

def temporal_feats(clip):
    T = len(clip)
    g = [cv2.resize(cv2.cvtColor(np.ascontiguousarray(x), cv2.COLOR_RGB2GRAY), (112, 112), interpolation=cv2.INTER_AREA) for x in clip]
    ycc = [cv2.cvtColor(np.ascontiguousarray(x), cv2.COLOR_RGB2YCrCb) for x in clip]
    L = {k: [] for k in ["dc", "dr", "mc", "mr", "ac", "ar", "vc", "vr", "rel"]}
    prev = None
    for t in range(T - 1):
        a, b = g[t], g[t + 1]
        diff = np.abs(b.astype(np.float32) - a.astype(np.float32))
        L["dc"].append(diff[_CEN].mean()); L["dr"].append(diff[_RING].mean())
        fl = cv2.calcOpticalFlowFarneback(a, b, None, 0.5, 2, 15, 2, 5, 1.1, 0)
        mg = np.hypot(fl[..., 0], fl[..., 1])
        L["mc"].append(mg[_CEN].mean()); L["mr"].append(mg[_RING].mean())
        dv = np.abs(np.gradient(fl[..., 0], axis=1) + np.gradient(fl[..., 1], axis=0))
        L["vc"].append(dv[_CEN].mean()); L["vr"].append(dv[_RING].mean())
        L["rel"].append(np.linalg.norm(fl[_CEN].mean(0) - fl[_RING].mean(0)))
        if prev is not None:
            ac = np.hypot(*(fl - prev).transpose(2, 0, 1)); L["ac"].append(ac[_CEN].mean()); L["ar"].append(ac[_RING].mean())
        prev = fl
    m = lambda k: float(np.mean(L[k])); s = lambda k: float(np.std(L[k]))
    f = {"t_diff_c": m("dc"), "t_diff_c_sd": s("dc"), "t_diff_r": m("dr"), "t_diff_r_sd": s("dr"),
         "t_diff_ratio": m("dc") / (m("dr") + 1e-6),
         "t_mag_c": m("mc"), "t_mag_c_sd": s("mc"), "t_mag_r": m("mr"), "t_mag_r_sd": s("mr"), "t_mag_ratio": m("mc") / (m("mr") + 1e-6),
         "t_acc_c": m("ac"), "t_acc_r": m("ar"), "t_acc_ratio": m("ac") / (m("ar") + 1e-6),
         "t_div_c": m("vc"), "t_div_r": m("vr"), "t_div_ratio": m("vc") / (m("vr") + 1e-6),
         "t_rel": m("rel"), "t_rel_sd": s("rel")}
    f["t_lum_sd"] = float(np.std([y[..., 0].mean() for y in ycc]))
    f["t_cr_sd"] = float(np.std([y[..., 1].mean() for y in ycc])); f["t_cb_sd"] = float(np.std([y[..., 2].mean() for y in ycc]))
    lap = [np.abs(cv2.Laplacian(x.astype(np.float32), cv2.CV_32F)).mean() for x in g]
    f["t_lap_cv"] = float(np.std(lap) / (np.mean(lap) + 1e-6))
    return f

def clip_features(clip):
    clip = np.asarray(clip)
    T = len(clip); idx = [0, T // 2, T - 1]
    fr = [frame_feats(np.ascontiguousarray(clip[i])) for i in idx]
    f = {k: float(np.mean([d[k] for d in fr])) for k in fr[0]}
    f.update(temporal_feats(clip))
    return f

def _feat_job(cid, variant, path):
    import cv2 as _cv2; _cv2.setNumThreads(1)
    d = clip_features(np.load(path)); d["clip_id"] = cid; d["variant"] = variant
    return d

def compute_features(meta):
    from joblib import Parallel, delayed
    jobs = [(r.clip_id, r.variant, P("clips", r.variant, r.clip_id + ".npy")) for r in meta.itertuples()]
    out = Parallel(n_jobs=CFG["NJOBS"], batch_size=8)(delayed(_feat_job)(*j) for j in tqdm(jobs, desc="features"))
    return meta.merge(pd.DataFrame(out), on=["clip_id", "variant"], how="left")

In [ ]:
FEAT_CSV = P("features.csv")
if os.path.exists(D("checkpoints", "features.csv")) and not os.path.exists(FEAT_CSV):
    shutil.copy(D("checkpoints", "features.csv"), FEAT_CSV)
if not os.path.exists(FEAT_CSV):
    FEAT = compute_features(META_ALL)
    FEAT.to_csv(FEAT_CSV, index=False); shutil.copy(FEAT_CSV, D("checkpoints", "features.csv"))
FEAT = pd.read_csv(FEAT_CSV)
FCOLS = [c for c in FEAT.columns if c[:2] in ("f_", "b_", "t_")]
GROUPS = {g: [c for c in FCOLS if c.startswith(g.lower() + "_")] for g in "FBT"}
FEAT[FCOLS] = FEAT[FCOLS].replace([np.inf, -np.inf], np.nan)
print({g: len(c) for g, c in GROUPS.items()}, "| rows:", len(FEAT), "| NaNs:", int(FEAT[FCOLS].isna().sum().sum()))

## Stage 5 — Classical (interpretable) detector: live vs. spoof

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.inspection import permutation_importance

KEYCOLS = ["clip_id", "video_id", "method", "label", "variant", "split"]

def _eer(y, p):
    fpr, tpr, _ = roc_curve(y, p); fnr = 1 - tpr; i = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[i] + fnr[i]) / 2)

def metrics(y, p, thr):
    y = np.asarray(y); p = np.asarray(p)
    if len(np.unique(y)) < 2: return dict(AUC=np.nan, EER=np.nan, HTER=np.nan, ACC=np.nan)
    pred = (p >= thr).astype(int)
    far = ((pred == 1) & (y == 0)).sum() / max((y == 0).sum(), 1)   # live rejected as spoof
    frr = ((pred == 0) & (y == 1)).sum() / max((y == 1).sum(), 1)   # spoof accepted as live
    return dict(AUC=roc_auc_score(y, p), EER=_eer(y, p), HTER=float((far + frr) / 2), ACC=float((pred == y).mean()))

def best_thr(y, p):
    y = np.asarray(y); p = np.asarray(p)
    ths = np.unique(np.quantile(p, np.linspace(0.01, 0.99, 197)))
    def hter(t):
        pred = p >= t
        return (((pred) & (y == 0)).sum() / max((y == 0).sum(), 1) + ((~pred) & (y == 1)).sum() / max((y == 1).sum(), 1)) / 2
    return float(ths[int(np.argmin([hter(t) for t in ths]))])

def eval_table(S, model, thr):
    d = S[S.split == "test"]
    if len(d) == 0 or d.label.nunique() < 2:
        return pd.DataFrame([dict(model=model, **metrics(d.label.values, d.prob.values, thr), AUC_video=np.nan)])
    m = metrics(d.label.values, d.prob.values, thr)
    vd = d.groupby("video_id").agg(label=("label", "first"), prob=("prob", "mean"))
    m["AUC_video"] = roc_auc_score(vd.label, vd.prob) if vd.label.nunique() > 1 else np.nan
    return pd.DataFrame([dict(model=model, **m)])

def train_hgb(df, cols):
    m = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.06, max_leaf_nodes=15, l2_regularization=1.0,
                                       class_weight="balanced", random_state=CFG["SEED"])
    m.fit(df[cols].values, df.label.values); return m

def score(m, df, cols):
    o = df[KEYCOLS].copy(); o["prob"] = m.predict_proba(df[cols].values)[:, 1]; return o

def run_model(name, cols):
    tr = FEAT[FEAT.split == "train"]
    va = FEAT[FEAT.split == "val"]
    m = train_hgb(tr, cols)
    sv = score(m, va, cols); thr = best_thr(sv.label.values, sv.prob.values)
    st = score(m, FEAT[FEAT.split == "test"], cols)
    return m, eval_table(st, name, thr), st, thr

In [ ]:
RES, SCORES, MODELS = [], {}, {}
ALLC = GROUPS["F"] + GROUPS["B"] + GROUPS["T"]
SETS = {"F": GROUPS["F"], "B": GROUPS["B"], "T": GROUPS["T"], "F+B": GROUPS["F"] + GROUPS["B"], "F+B+T": ALLC}
for gname, cols in SETS.items():
    name = f"HGB[{gname}]"
    m, tbl, st, thr = run_model(name, cols)
    RES.append(tbl); SCORES[name] = st
    if gname == "F+B+T": MODELS["hgb"] = (m, cols, thr)
print("trained", len(SCORES), "classical models (feature-group ablation)")

CLASSICAL = pd.concat(RES, ignore_index=True)
CLASSICAL.to_csv(D("results", "classical_results.csv"), index=False)
print("\nClassical model results (test set):")
print(CLASSICAL.round(3).to_string(index=False))

# Permutation importance on the full-feature model, so we know *which*
# cues actually separate live faces from print/replay spoofs
m_full, cols_full, thr_full = MODELS["hgb"]
va = FEAT[FEAT.split == "val"]
pi = permutation_importance(m_full, va[cols_full].values, va.label.values, n_repeats=5,
                             random_state=CFG["SEED"], scoring="roc_auc", n_jobs=-1)
TOPF = pd.Series(pi.importances_mean, index=cols_full).sort_values(ascending=False).head(15)
print("\nTop 15 features by permutation importance:")
print(TOPF.round(4).to_string())

## Stage 6 — Deep baseline (EfficientNet-B0 fine-tune) on the same clips

In [ ]:
import torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader
DEV = "cuda" if torch.cuda.is_available() else "cpu"
MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)
BASE = META_ALL[META_ALL.variant == "clean"].reset_index(drop=True)

class FrameDS(Dataset):
    def __init__(self, clips, train, frames=(0, 2, 4, 6), reps=1):
        self.clips = clips.reset_index(drop=True); self.train = train
        self.frames = frames; self.reps = reps
        self.n = len(self.clips) * (reps if train else len(frames))
    def __len__(self): return self.n
    def __getitem__(self, i):
        if self.train:
            ci = i % len(self.clips); t = random.randrange(CFG["T"])
        else:
            ci = i // len(self.frames); t = self.frames[i % len(self.frames)]
        r = self.clips.iloc[ci]
        x = np.asarray(np.load(P("clips", "clean", r.clip_id + ".npy"), mmap_mode="r")[t], np.float32) / 255.0
        if self.train and random.random() < 0.5: x = x[:, ::-1]
        x = torch.from_numpy(np.ascontiguousarray(((x - MEAN) / STD).transpose(2, 0, 1)))
        return x, torch.tensor(float(r.label)), ci

def _amp(): return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=False)  # fp32 (fp16 autocast caused NaNs previously)

@torch.no_grad()
def predict(model, clips, bs=128):
    dl = DataLoader(FrameDS(clips, False), batch_size=bs, num_workers=2)
    model.eval(); L = np.zeros(len(clips)); C = np.zeros(len(clips))
    for x, _, ci in dl:
        with _amp(): o = model(x.to(DEV)).squeeze(1).float().cpu().numpy()
        np.add.at(L, ci.numpy(), o); np.add.at(C, ci.numpy(), 1)
    return 1 / (1 + np.exp(-L / np.maximum(C, 1)))

def clip_scores(model, clips):
    o = clips[KEYCOLS].copy().reset_index(drop=True); o["prob"] = predict(model, clips); return o

def train_cnn(name, epochs=8, reps=6, bs=64, lr=2e-4):
    trc = BASE[BASE.split == "train"]; vac = BASE[BASE.split == "val"]
    dl = DataLoader(FrameDS(trc, True, reps=reps), batch_size=bs, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
    model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=1).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(dl), pct_start=0.15)
    pw = torch.tensor([(trc.label == 0).sum() / max((trc.label == 1).sum(), 1)], device=DEV, dtype=torch.float32)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    try: scaler = torch.amp.GradScaler("cuda", enabled=False)
    except Exception: scaler = torch.cuda.amp.GradScaler(enabled=False)
    best, ckpt = -1, P("models", name + ".pt")
    for ep in range(epochs):
        model.train(); tl = 0
        for x, y, _ in tqdm(dl, desc=f"{name} ep{ep + 1}/{epochs}", leave=False):
            x, y = x.to(DEV, non_blocking=True), y.to(DEV)
            with _amp(): out = model(x).squeeze(1)
            loss = crit(out.float(), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); tl += loss.item()
        s = clip_scores(model, vac); va_auc = roc_auc_score(s.label, s.prob)
        print(f"  ep{ep + 1}: train_loss={tl / len(dl):.3f}  val_AUC={va_auc:.3f}")
        if va_auc > best: best = va_auc; torch.save(model.state_dict(), ckpt)
    model.load_state_dict(torch.load(ckpt, map_location=DEV)); shutil.copy(ckpt, D("checkpoints", name + ".pt"))
    return model

def eval_cnn(model, name):
    tec = BASE[BASE.split == "test"]
    vac = BASE[BASE.split == "val"]
    sv = clip_scores(model, vac); thr = best_thr(sv.label.values, sv.prob.values)
    st = clip_scores(model, tec)
    return eval_table(st, name, thr), st, thr

In [ ]:
cnn_model = train_cnn("cnn_liveness", epochs=8, reps=6)
cnn_tbl, cnn_scores, cnn_thr = eval_cnn(cnn_model, "CNN[EffNet-B0]")
CNN = cnn_tbl.copy()
CNN.to_csv(D("results", "cnn_results.csv"), index=False)
print("\nCNN results (test set):")
print(CNN.round(3).to_string(index=False))

## Stage 7 — Auto-generated results file for your documentation

In [ ]:
def md_table(df, floatfmt=3):
    df = df.copy()
    cols = [" / ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in df.columns]
    L = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        L.append("| " + " | ".join([f"{v:.{floatfmt}f}" if isinstance(v, (float, np.floating)) else str(v) for v in r.values]) + " |")
    return "\n".join(L)

ALLRES = pd.concat([CLASSICAL, CNN], ignore_index=True) if "CNN" in globals() else CLASSICAL
ALLRES.to_csv(D("results", "all_results.csv"), index=False)

cnt_long = META_ALL[META_ALL.variant == "clean"].groupby(["method", "split"]).agg(
    clips=("clip_id", "count"), videos=("video_id", "nunique")).reset_index()
cnt = cnt_long.pivot(index="method", columns="split", values=["clips", "videos"]).fillna(0).astype(int)
cnt.to_csv(D("results", "dataset_summary.csv"))

md = [
    "# SiW Live vs. Spoof — Results summary (auto-generated)\n",
    f"Config: {json.dumps({k: v for k, v in CFG.items() if k in ['T', 'SIZE', 'CLIPS_PER_VIDEO', 'SEED', 'TRAIN_FRAC', 'VAL_FRAC']})}\n",
    "## Dataset actually used (clips / videos per split, Live vs Spoof)\n", md_table(cnt_long), "\n",
    "## Test-set results — all models (AUC, EER, HTER, accuracy)\n", md_table(ALLRES), "\n",
    "## Feature-group ablation (classical model only)\n",
    md_table(CLASSICAL[["model", "AUC", "HTER", "ACC"]]), "\n",
    "## Top 15 features by permutation importance (full-feature classical model)\n",
    md_table(TOPF.round(4).to_frame("perm_importance").reset_index().rename(columns={"index": "feature"})), "\n",
]
open(D("results", "results_summary.md"), "w").write("\n".join(md))
print("✅ Everything saved to:", D("results"))
print(sorted(os.listdir(D("results"))))

In [ ]:
# Download everything (tables + summary) to your laptop as one zip
import shutil as _sh
zpath = "/content/siw_results_bundle"
_sh.make_archive(zpath, "zip", D("results"))
from google.colab import files
files.download(zpath + ".zip")